# NGCF · 개인7시드+학교2시드+기존49 → 최종10시드 합산

학습·test 재평가 없이 원본arm JSON 전체 정밀도로 합산합니다. 학습조건·입력·코드·seed49 원본 지문이 다르거나 중복시드가 있으면 중단합니다. 두 계정의 원래 run identity를 그대로 보존합니다. 80개 미만이면 최종평균을 만들지 않습니다. CPU 런타임으로 충분합니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys, torch
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
# GPU 학습 없음
GPU = 'CPU aggregation'

SOURCE_COMMIT = '63146eff4ed60bfb0cc430b77d58fdc40e4db7c5'
REPO = Path('/content/clv-dh-ngcf-remaining9-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', SOURCE_COMMIT], check=True)
actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == SOURCE_COMMIT
sys.path.insert(0, str(REPO))
print('GPU:', GPU, '| 고정 코드:', SOURCE_COMMIT)
import hashlib, urllib.request, importlib.util
HELPER_COMMIT = '78380e4bca3a293be28bab5336ae9b16feffb7b6'
HELPERS = {
    'clv_dh_ngcf_split72': '96c0b3d9a08d2a244c1bc15ea1967fa3414b85f46754cdb69ad9bfeba2b5fb43',
    'clv_ngcf_school_storage': '11d8b6dba321cfb72da1715208df4a45adf4495625394ff7f1d72fc25bebe0d4',
}
for name, expected in HELPERS.items():
    url = f'https://raw.githubusercontent.com/jung-un/clv-m2-lightgcn-runner/{HELPER_COMMIT}/{name}.py'
    with urllib.request.urlopen(url, timeout=60) as response:
        data = response.read()
    assert hashlib.sha256(data).hexdigest() == expected
    target = Path('/content') / (name + '.py')
    target.write_bytes(data)
    spec = importlib.util.spec_from_file_location(name, target)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
import clv_dh_ngcf_split72 as split
import clv_ngcf_school_storage as school_storage
print('고정 학습코드63146eff / 분담도우미:', HELPER_COMMIT)


## 1. 두 결과 ZIP 선택
같은 계정에서 파일을 볼 수 없으면 Mac을 통해 전달합니다. checkpoint는 업로드할 필요가 없습니다.

In [ ]:
from google.colab import files
files.upload()
PERSONAL = Path('/content/dh_ngcf_personal7_results.zip')
SCHOOL = Path('/content/dh_ngcf_school2_results.zip')
assert PERSONAL.is_file() and SCHOOL.is_file()


## 2. 중복·누락 검증, 10시드 평균·표본표준편차
표준편차는 시드 변동성이며 유의성 검정이 아닙니다.

In [ ]:
destination = Path('/content/drive/MyDrive/논문/data/results_v3_dunnhumby_ngcf_split72_merged_v1')
result = split.merge(PERSONAL, SCHOOL, destination)
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))
if result['reading']['complete']:
    display(result['core_mean_std'])
else:
    print('누락 모델/시드를 완료한 패킷을 회수하세요. 추가 학습/재평가하지 않습니다.')
